# Predicting BDD100K Detector Failures Across Validation Slices

This notebook evaluates whether OOD scores help predict failures for a COCO-pretrained object detector on BDD100K. BDD100K does not include a separate BDD100K-O split, so we use clear daytime BDD100K train images as the in-distribution reference set and partition BDD100K validation into reference-like clear daytime images and shifted non-clear-daytime images.

Each row is an image-class pair, so the notebook asks whether the detector is likely to fail for a specific class in a specific driving scene. The goal is a focused demo for this detector, dataset, and validation partition. Detector confidence is expected to be a strong baseline; we test whether global image OOD and object-region/chip OOD add complementary signal.

## What This Demonstrates

- Build image-class rows from detections and ground truth.
- Partition BDD100K validation by weather/time attributes into reference-like and shifted slices.
- Compute detector confidence, global OOD, and chip-level OOD features.
- Compare failure-risk models and inspect the highest-risk image-class examples visually.

In [ ]:
import os
from pathlib import Path

DATASETS_ROOT = Path(os.environ.get('OODKIT_DATASETS', '/datasets'))
BDD_ROOT = Path(os.environ.get('OODKIT_BDD_ROOT', DATASETS_ROOT / 'bdd100k'))
BDD_IMAGE_ROOT = Path(os.environ.get('OODKIT_BDD_IMAGE_ROOT', BDD_ROOT / 'images' / '100k'))
BDD_LABEL_ROOT = Path(os.environ.get('OODKIT_BDD_LABEL_ROOT', BDD_ROOT / 'labels'))

BACKBONE = os.environ.get('OODKIT_BACKBONE', 'dinov2-small')
DETECTOR_MODEL = os.environ.get('OODKIT_DETECTOR_MODEL', 'fasterrcnn_resnet50_fpn_v2')
DETECTION_SCORE_THRESHOLD = float(os.environ.get('OODKIT_DETECTION_SCORE_THRESHOLD', '0.25'))
IOU_THRESHOLD = float(os.environ.get('OODKIT_IOU_THRESHOLD', '0.5'))
HEAD_EPOCHS = int(os.environ.get('OODKIT_HEAD_EPOCHS', '3'))
HEAD_LR = float(os.environ.get('OODKIT_HEAD_LR', '1e-3'))
VIM_PCT_VARIANCE = float(os.environ.get('OODKIT_VIM_PCT_VARIANCE', '0.95'))

MAX_TRAIN_REFERENCE_IMAGES = int(os.environ.get('OODKIT_MAX_TRAIN_REFERENCE_IMAGES', '5000'))
MAX_VAL_REFERENCE_IMAGES = int(os.environ.get('OODKIT_MAX_VAL_REFERENCE_IMAGES', '1500'))
MAX_VAL_SHIFTED_IMAGES = int(os.environ.get('OODKIT_MAX_VAL_SHIFTED_IMAGES', '1500'))

BATCH_SIZE = int(os.environ.get('OODKIT_BATCH_SIZE', '32'))
DETECTOR_BATCH_SIZE = int(os.environ.get('OODKIT_DETECTOR_BATCH_SIZE', '4'))
NUM_WORKERS = int(os.environ.get('OODKIT_NUM_WORKERS', '4'))
PIN_MEMORY = None
PERSISTENT_WORKERS = NUM_WORKERS > 0
HIGH_CONFIDENCE_THRESHOLD = float(os.environ.get('OODKIT_HIGH_CONFIDENCE_THRESHOLD', '0.75'))

DEFAULT_CHECKPOINT_PATH = Path('notebooks/outputs/bdd100k_embedder_checkpoint')
if not DEFAULT_CHECKPOINT_PATH.exists():
    DEFAULT_CHECKPOINT_PATH = Path('outputs/bdd100k_embedder_checkpoint')
CHECKPOINT_PATH = Path(os.environ.get('OODKIT_BDD_FAILURE_CHECKPOINT', str(DEFAULT_CHECKPOINT_PATH)))
USE_CHECKPOINT = os.environ.get('OODKIT_USE_BDD_CHECKPOINT', '1') != '0'
REFERENCE_DATASET_NAME = 'BDD100K-val-reference'
SHIFTED_DATASET_NAME = 'BDD100K-val-shifted'
SEED = 7

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from torch.utils.data import Dataset

from oodkit.data.chip_dataset import ChipDataset
from oodkit.detection import (
    attach_ood_features,
    detection_chips_from_table,
    evaluate_detection_tables,
    run_torchvision_detector,
)
from oodkit.detectors import ViM
from oodkit.embeddings.embedder import Embedder
from oodkit.failure import calibration_bins, evaluate_failure_baselines

rng = np.random.default_rng(SEED)

In [ ]:
def first_existing(paths, *, label):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    candidates = '\n'.join(f'  - {Path(p)}' for p in paths)
    raise FileNotFoundError(f'Could not find {label}. Checked:\n{candidates}')


TRAIN_LABELS = Path(os.environ['OODKIT_BDD_TRAIN_LABELS']) if 'OODKIT_BDD_TRAIN_LABELS' in os.environ else first_existing(
    [
        BDD_LABEL_ROOT / 'det_20' / 'det_train.json',
        BDD_LABEL_ROOT / 'bdd100k_labels_images_train.json',
        BDD_LABEL_ROOT / 'det_train.json',
    ],
    label='BDD100K train detection labels',
)
VAL_LABELS = Path(os.environ['OODKIT_BDD_VAL_LABELS']) if 'OODKIT_BDD_VAL_LABELS' in os.environ else first_existing(
    [
        BDD_LABEL_ROOT / 'det_20' / 'det_val.json',
        BDD_LABEL_ROOT / 'bdd100k_labels_images_val.json',
        BDD_LABEL_ROOT / 'det_val.json',
    ],
    label='BDD100K val detection labels',
)
TRAIN_IMAGES = Path(os.environ.get('OODKIT_BDD_TRAIN_IMAGES', BDD_IMAGE_ROOT / 'train'))
VAL_IMAGES = Path(os.environ.get('OODKIT_BDD_VAL_IMAGES', BDD_IMAGE_ROOT / 'val'))

for path in [TRAIN_LABELS, VAL_LABELS, TRAIN_IMAGES, VAL_IMAGES]:
    if not Path(path).exists():
        raise FileNotFoundError(path)

print(f'Train labels: {TRAIN_LABELS}')
print(f'Val labels:   {VAL_LABELS}')
print(f'Train images: {TRAIN_IMAGES}')
print(f'Val images:   {VAL_IMAGES}')

## BDD100K Loading And Validation Partition

The reference slice is clear daytime driving. The shifted slice is everything else in BDD100K validation, including night, dawn/dusk, overcast, rainy, snowy, foggy, and undefined conditions. This is a pragmatic split for a demo, not a claim that the shifted slice is a complete deployment benchmark.

In [ ]:
BDD_CLASS_ORDER = [
    'bike',
    'bus',
    'car',
    'motor',
    'person',
    'rider',
    'traffic light',
    'traffic sign',
    'train',
    'truck',
]


def load_bdd_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def bdd_partition(record):
    attrs = record.get('attributes') or {}
    weather = str(attrs.get('weather', 'undefined')).lower()
    timeofday = str(attrs.get('timeofday', 'undefined')).lower()
    if weather == 'clear' and timeofday == 'daytime':
        return 'reference'
    return 'shifted'


def bdd_attributes(record):
    attrs = record.get('attributes') or {}
    return {
        'weather': str(attrs.get('weather', 'undefined')).lower(),
        'timeofday': str(attrs.get('timeofday', 'undefined')).lower(),
        'scene': str(attrs.get('scene', 'undefined')).lower(),
    }


def bdd_class_names(label_paths):
    names = set()
    for path in label_paths:
        for rec in load_bdd_json(path):
            for obj in rec.get('labels', []):
                if obj.get('box2d') is not None and obj.get('category'):
                    names.add(str(obj['category']))
    ordered = [name for name in BDD_CLASS_ORDER if name in names]
    ordered.extend(sorted(names.difference(ordered)))
    return ordered


def sample_records(records, max_images, *, seed):
    if max_images is None or len(records) <= max_images:
        return list(records)
    keep = set(np.random.default_rng(seed).choice(len(records), size=max_images, replace=False))
    return [rec for i, rec in enumerate(records) if i in keep]


def read_image_size(path):
    with Image.open(path) as img:
        return int(img.width), int(img.height)


def bdd_tables(
    label_json,
    image_root,
    class_to_idx,
    class_names,
    *,
    dataset_name,
    image_id_prefix,
    partition=None,
    max_images=None,
    seed=0,
):
    records = load_bdd_json(label_json)
    if partition is not None:
        records = [rec for rec in records if bdd_partition(rec) == partition]
    records = sample_records(records, max_images, seed=seed)

    image_rows = []
    gt_rows = []
    missing_images = 0
    for rec in records:
        image_name = rec.get('name')
        if not image_name:
            continue
        image_path = Path(image_root) / image_name
        if not image_path.exists():
            missing_images += 1
            continue
        width, height = read_image_size(image_path)
        attrs = bdd_attributes(rec)
        public_id = f'{image_id_prefix}:{Path(image_name).stem}'
        image_rows.append(
            {
                'image_id': public_id,
                'source_image_id': Path(image_name).stem,
                'file_name': image_name,
                'image_path': str(image_path),
                'image_width': width,
                'image_height': height,
                'dataset_name': dataset_name,
                'bdd_partition': bdd_partition(rec),
                **attrs,
            }
        )

        for obj_idx, obj in enumerate(rec.get('labels', [])):
            box = obj.get('box2d')
            category = obj.get('category')
            if box is None or category not in class_to_idx:
                continue
            x1 = float(box['x1'])
            y1 = float(box['y1'])
            x2 = float(box['x2'])
            y2 = float(box['y2'])
            if x2 <= x1 or y2 <= y1:
                continue
            class_id = int(class_to_idx[str(category)])
            gt_rows.append(
                {
                    'image_id': public_id,
                    'class_id': class_id,
                    'class_name': class_names[class_id],
                    'bbox': [x1, y1, x2, y2],
                    'gt_id': f'{public_id}_gt_{obj.get("id", obj_idx)}',
                    'image_path': str(image_path),
                    'image_width': width,
                    'image_height': height,
                    'dataset_name': dataset_name,
                    'bdd_partition': bdd_partition(rec),
                    **attrs,
                }
            )

    if missing_images:
        print(f'[warn] {label_json.name}: skipped {missing_images} missing images')
    return pd.DataFrame(image_rows), pd.DataFrame(gt_rows)


class ImagePathDataset(Dataset):
    def __init__(self, frame, processor):
        self.frame = frame.reset_index(drop=True)
        self.processor = processor
        self.imgs = [(p, -1) for p in self.frame['image_path'].tolist()]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path = self.frame.iloc[index]['image_path']
        with Image.open(path) as img:
            img = img.convert('RGB')
            return self.processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)


def image_maps(image_frame):
    paths = dict(zip(image_frame['image_id'], image_frame['image_path']))
    sizes = dict(zip(image_frame['image_id'], zip(image_frame['image_width'], image_frame['image_height'])))
    return paths, sizes


def add_reference_zscores(frame, score_cols, *, reference_domain=REFERENCE_DATASET_NAME):
    out = frame.copy()
    ref_mask = out['dataset_name'].eq(reference_domain)
    for col in score_cols:
        ref = pd.to_numeric(out.loc[ref_mask, col], errors='coerce')
        mean = ref.mean()
        std = ref.std(ddof=0)
        if not np.isfinite(std) or std == 0:
            std = 1.0
        out[f'{col}_z'] = (pd.to_numeric(out[col], errors='coerce') - mean) / std
    out['combined_ood_z'] = out[[f'{c}_z' for c in score_cols]].fillna(0.0).sum(axis=1)
    return out


def classification_metrics(y_true, y_prob):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(y_prob, dtype=float)
    if len(np.unique(y)) < 2:
        return {'AUROC': np.nan, 'AUPR': np.nan, 'Brier': brier_score_loss(y, p)}
    return {
        'AUROC': roc_auc_score(y, p),
        'AUPR': average_precision_score(y, p),
        'Brier': brier_score_loss(y, p),
    }


def prediction_metrics_by_domain(predictions, features):
    pred = predictions.merge(
        features[['image_id', 'class_id', 'dataset_name']],
        on=['image_id', 'class_id'],
        how='left',
    )
    rows = []
    for (model, dataset_name), group in pred.groupby(['model', 'dataset_name'], sort=True):
        metric = classification_metrics(group['target'], group['predicted_failure_probability'])
        rows.append({'model': model, 'dataset_name': dataset_name, 'n': len(group), **metric})
    return pd.DataFrame(rows)


def confidence_ood_failure_table(
    frame,
    *,
    confidence_col='mean_detection_confidence',
    ood_col='combined_ood_z',
    confidence_bins=(0.5, 0.7, 0.9, 1.01),
):
    rows = []
    work = frame[np.isfinite(frame[confidence_col]) & np.isfinite(frame[ood_col])].copy()
    for lo, hi in zip(confidence_bins[:-1], confidence_bins[1:]):
        group = work[(work[confidence_col] >= lo) & (work[confidence_col] < hi)]
        if len(group) < 4:
            continue
        low_cut = group[ood_col].quantile(0.33)
        high_cut = group[ood_col].quantile(0.67)
        low = group[group[ood_col] <= low_cut]
        high = group[group[ood_col] >= high_cut]
        rows.append(
            {
                'confidence_bin': f'{lo:0.1f}-{min(hi, 1.0):0.1f}',
                'n': len(group),
                'low_ood_n': len(low),
                'low_ood_failure_rate': low['has_failure'].mean(),
                'high_ood_n': len(high),
                'high_ood_failure_rate': high['has_failure'].mean(),
            }
        )
    return pd.DataFrame(rows)


def print_metrics(metrics):
    display(metrics.sort_values('AUROC', ascending=False).reset_index(drop=True))

In [ ]:
class_names_list = bdd_class_names([TRAIN_LABELS, VAL_LABELS])
class_to_idx = {name: idx for idx, name in enumerate(class_names_list)}
class_names = dict(enumerate(class_names_list))

train_images, train_gt = bdd_tables(
    TRAIN_LABELS,
    TRAIN_IMAGES,
    class_to_idx,
    class_names,
    dataset_name='BDD100K-train-reference',
    image_id_prefix='train-ref',
    partition='reference',
    max_images=MAX_TRAIN_REFERENCE_IMAGES,
    seed=SEED,
)
val_ref_images, val_ref_gt = bdd_tables(
    VAL_LABELS,
    VAL_IMAGES,
    class_to_idx,
    class_names,
    dataset_name=REFERENCE_DATASET_NAME,
    image_id_prefix='val-ref',
    partition='reference',
    max_images=MAX_VAL_REFERENCE_IMAGES,
    seed=SEED + 1,
)
val_shift_images, val_shift_gt = bdd_tables(
    VAL_LABELS,
    VAL_IMAGES,
    class_to_idx,
    class_names,
    dataset_name=SHIFTED_DATASET_NAME,
    image_id_prefix='val-shift',
    partition='shifted',
    max_images=MAX_VAL_SHIFTED_IMAGES,
    seed=SEED + 2,
)

if train_images.empty or train_gt.empty:
    raise RuntimeError('No BDD100K reference training images with GT boxes were found.')
if val_ref_images.empty or val_shift_images.empty:
    raise RuntimeError('Both reference and shifted BDD100K validation slices must contain images.')

eval_images = pd.concat([val_ref_images, val_shift_images], ignore_index=True)
eval_gt = pd.concat([val_ref_gt, val_shift_gt], ignore_index=True)

print(f'BDD classes ({len(class_names_list)}): {class_names_list}')
print(f'Train reference images: {len(train_images)} | GT boxes: {len(train_gt)}')
print(f'Val reference images:   {len(val_ref_images)} | GT boxes: {len(val_ref_gt)}')
print(f'Val shifted images:     {len(val_shift_images)} | GT boxes: {len(val_shift_gt)}')
display(eval_images.groupby(['dataset_name', 'timeofday', 'weather']).size().rename('images').reset_index().head(20))

## Run Detector And Build Image-Class Targets

A row is marked as failed if the detector has any false positive or false negative for that class in that image. GT-only rows represent missed classes. Detection-only rows represent hallucinated classes. Predictions are mapped from torchvision/COCO labels into the BDD100K detection vocabulary where possible.

In [ ]:
TORCHVISION_TO_BDD = {
    'person': 'person',
    'bicycle': 'bike',
    'car': 'car',
    'motorcycle': 'motor',
    'bus': 'bus',
    'truck': 'truck',
    'train': 'train',
    'traffic light': 'traffic light',
    'stop sign': 'traffic sign',
}
detector_category_map = {
    tv_name: class_to_idx[bdd_name]
    for tv_name, bdd_name in TORCHVISION_TO_BDD.items()
    if bdd_name in class_to_idx
}


def normalize_bdd_predictions(pred):
    if pred.empty:
        return pred
    out = pred[pred['class_name'].isin(detector_category_map)].copy()
    out['class_id'] = out['class_name'].map(detector_category_map).astype(int)
    out['class_name'] = out['class_id'].map(class_names)
    return out.reset_index(drop=True)


pred_ref = run_torchvision_detector(
    val_ref_images['image_path'].tolist(),
    image_ids=val_ref_images['image_id'].tolist(),
    category_table=detector_category_map,
    model_name=DETECTOR_MODEL,
    score_threshold=DETECTION_SCORE_THRESHOLD,
    batch_size=DETECTOR_BATCH_SIZE,
    detector_name=DETECTOR_MODEL,
    dataset_name=REFERENCE_DATASET_NAME,
)
pred_shift = run_torchvision_detector(
    val_shift_images['image_path'].tolist(),
    image_ids=val_shift_images['image_id'].tolist(),
    category_table=detector_category_map,
    model_name=DETECTOR_MODEL,
    score_threshold=DETECTION_SCORE_THRESHOLD,
    batch_size=DETECTOR_BATCH_SIZE,
    detector_name=DETECTOR_MODEL,
    dataset_name=SHIFTED_DATASET_NAME,
)
predictions = pd.concat([normalize_bdd_predictions(pred_ref), normalize_bdd_predictions(pred_shift)], ignore_index=True)

tables = evaluate_detection_tables(
    predictions,
    eval_gt,
    iou_threshold=IOU_THRESHOLD,
    backend='auto',
    class_names=class_names,
    detector_name=DETECTOR_MODEL,
)

summary = (
    tables.image_class_metrics.groupby('dataset_name')
    .agg(rows=('image_id', 'size'), failure_rate=('has_failure', 'mean'), detections=('num_detections', 'sum'), gt=('num_gt', 'sum'))
    .reset_index()
)
display(summary)
display(tables.image_class_metrics.head())

## Fit ViM Reference Scores

ViM needs logits, so this notebook trains or loads a lightweight BDD100K chip/category head. The same head is reused for full images and detection chips, which keeps the OOD scoring method consistent across scales.

In [ ]:
if USE_CHECKPOINT and CHECKPOINT_PATH.exists():
    print(f'Loading checkpoint from {CHECKPOINT_PATH}')
    embedder = Embedder.load(CHECKPOINT_PATH)
else:
    embedder = Embedder(backbone=BACKBONE)

processor = embedder._processor  # noqa: SLF001 - notebook research workflow
_DL_KW = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS)

train_paths, train_sizes = image_maps(train_images)
eval_paths, eval_sizes = image_maps(eval_images)

train_chip_source = train_gt.rename(columns={'gt_id': 'detection_id'})
train_chip_anns, _ = detection_chips_from_table(
    train_chip_source,
    image_paths=train_paths,
    image_sizes=train_sizes,
)
train_chip_ds = ChipDataset(train_chip_anns, processor, class_names=class_names_list)

if embedder._head is None:  # noqa: SLF001
    embedder.fit(
        train_chip_ds,
        mode='head',
        epochs=HEAD_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=HEAD_LR,
        save=False,
        **_DL_KW,
    )

head = embedder._head  # noqa: SLF001 - notebook research workflow
assert head is not None
W = head.weight.detach().cpu().numpy()
b = head.bias.detach().cpu().numpy()

In [ ]:
train_image_ds = ImagePathDataset(train_images, processor)
eval_image_ds = ImagePathDataset(eval_images, processor)
train_global = embedder.extract(train_image_ds, batch_size=BATCH_SIZE, **_DL_KW)
eval_global = embedder.extract(eval_image_ds, batch_size=BATCH_SIZE, **_DL_KW)
assert train_global.logits is not None and eval_global.logits is not None

global_vim = ViM(W, b, pct_variance=VIM_PCT_VARIANCE).fit(train_global.to_features())
global_scores = pd.DataFrame(
    {
        'image_id': eval_images['image_id'].tolist(),
        'global_ood_score': global_vim.score(eval_global.to_features()),
    }
)

train_chip_res = embedder.extract(train_chip_ds, batch_size=BATCH_SIZE, **_DL_KW)
assert train_chip_res.logits is not None and train_chip_res.labels is not None
chip_vim = ViM(W, b, pct_variance=VIM_PCT_VARIANCE).fit(train_chip_res.to_features())

det_enriched = tables.detections_enriched.copy()
if len(det_enriched):
    chip_anns, ordered_detection_ids = detection_chips_from_table(
        det_enriched,
        image_paths=eval_paths,
        image_sizes=eval_sizes,
    )
    pred_chip_ds = ChipDataset(chip_anns, processor, class_names=class_names_list)
    pred_chip_res = embedder.extract(pred_chip_ds, batch_size=BATCH_SIZE, **_DL_KW)
    assert pred_chip_res.logits is not None
    chip_score_frame = pd.DataFrame(
        {
            'detection_id': ordered_detection_ids,
            'chip_ood_score': chip_vim.score(pred_chip_res.to_features()),
        }
    )
    det_enriched = det_enriched.merge(chip_score_frame, on='detection_id', how='left')
else:
    det_enriched['chip_ood_score'] = []

image_class_features = attach_ood_features(
    tables.image_class_metrics,
    global_scores=global_scores,
    detections_enriched=det_enriched,
    embedding_model_name=BACKBONE,
    ood_detector_name='ViM',
)
image_class_features = add_reference_zscores(
    image_class_features,
    ['global_ood_score', 'chip_ood_mean'],
    reference_domain=REFERENCE_DATASET_NAME,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

display(image_class_features.groupby('dataset_name')[['global_ood_score', 'chip_ood_mean', 'combined_ood_z']].mean().reset_index())
display(
    image_class_features.assign(
        gt_only=lambda x: (x['num_gt'] > 0) & (x['num_detections'] == 0),
        detection_only=lambda x: (x['num_gt'] == 0) & (x['num_detections'] > 0),
        missing_chip_features=lambda x: x['num_chips'] == 0,
    )
    .groupby('dataset_name')
    .agg(gt_only=('gt_only', 'sum'), detection_only=('detection_only', 'sum'), missing_chip_features=('missing_chip_features', 'sum'))
    .reset_index()
)

## Failure Prediction Baselines

These baselines are deliberately simple and use only inference-time features.

- `confidence only`: detector confidence summaries and detection count.
- `OOD only`: global image ViM and detection-chip ViM summaries.
- `confidence + OOD`: combines detector confidence with global and chip OOD features.
- `confidence + OOD + class`: adds class identity as a categorical prior.

Target-derived columns such as `tp`, `fp`, `fn`, `precision`, `recall`, and `has_failure` are evaluation outputs only and must not be used as model inputs.

In [ ]:
CONFIDENCE_FEATURES = [
    'mean_detection_confidence',
    'max_detection_confidence',
    'min_detection_confidence',
    'num_detections',
]
OOD_FEATURES = [
    'global_ood_score',
    'chip_ood_mean',
    'chip_ood_max',
    'chip_ood_p90',
    'chip_ood_std',
    'num_chips',
]
FEATURE_SETS = {
    'failure prior': [],
    'class identity only': ['class_id'],
    'confidence only': CONFIDENCE_FEATURES,
    'OOD only': OOD_FEATURES,
    'confidence + OOD': CONFIDENCE_FEATURES + OOD_FEATURES,
    'confidence + OOD + class': CONFIDENCE_FEATURES + OOD_FEATURES + ['class_id'],
}

TARGET_DERIVED_COLUMNS = {'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'has_failure'}
MODEL_INPUT_COLUMNS = {col for cols in FEATURE_SETS.values() for col in cols}
assert MODEL_INPUT_COLUMNS.isdisjoint(TARGET_DERIVED_COLUMNS)

metrics, failure_predictions = evaluate_failure_baselines(
    image_class_features,
    target_col='has_failure',
    feature_sets=FEATURE_SETS,
    group_col='image_id',
    test_size=0.3,
    random_state=SEED,
)
print_metrics(metrics)

print('Held-out prediction metrics by validation slice:')
display(prediction_metrics_by_domain(failure_predictions, image_class_features).sort_values(['dataset_name', 'AUROC'], ascending=[True, False]))

In this BDD100K partition experiment, detector confidence is the strongest baseline to beat. OOD-only features are useful if they are meaningfully predictive on the shifted validation slice, and the key comparison is confidence-only vs confidence + OOD + class.

In [ ]:
best_model = metrics.sort_values('AUROC', ascending=False).iloc[0]['model']
pred_best = failure_predictions[failure_predictions['model'] == best_model].copy()
cal = calibration_bins(pred_best['target'], pred_best['predicted_failure_probability'], n_bins=10)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], color='0.7', linestyle='--')
ax.scatter(cal['mean_predicted'], cal['observed_rate'], s=np.maximum(cal['count'], 1) * 8)
ax.set_title(f'Calibration: {best_model}')
ax.set_xlabel('Mean predicted failure probability')
ax.set_ylabel('Observed failure rate')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()
display(cal)

## High-Confidence Residual Risk

Overall AUROC can hide the value of OOD features because confidence catches many obvious failures. In this experiment, the more relevant question is whether high-OOD rows fail more often among rows with similar confidence, especially in the shifted validation slice.

In [ ]:
high_conf_failures = image_class_features[
    image_class_features['mean_detection_confidence'].ge(HIGH_CONFIDENCE_THRESHOLD)
    & image_class_features['has_failure'].astype(bool)
].sort_values('combined_ood_z', ascending=False)

display(
    high_conf_failures[
        [
            'dataset_name',
            'image_id',
            'class_name',
            'num_gt',
            'num_detections',
            'tp',
            'fp',
            'fn',
            'mean_detection_confidence',
            'global_ood_score_z',
            'chip_ood_mean_z',
            'combined_ood_z',
        ]
    ].head(20)
)

print('Same-confidence failure rates split by OOD terciles, shifted slice only:')
display(
    confidence_ood_failure_table(
        image_class_features[image_class_features['dataset_name'].eq(SHIFTED_DATASET_NAME)]
    )
)

print('Same-confidence failure rates split by OOD terciles, all eval rows:')
display(confidence_ood_failure_table(image_class_features))

The high-confidence shifted slice is the main residual-risk view for this run: OOD is most interesting when it identifies suspicious image-class rows where this detector still appears confident.

## Risk Table And Qualitative Examples

In [ ]:
risk = pred_best.merge(
    image_class_features,
    on=['image_id', 'class_id', 'class_name'],
    how='left',
)
risk = risk.sort_values('predicted_failure_probability', ascending=False)
display(
    risk[
        [
            'dataset_name',
            'image_id',
            'class_name',
            'target',
            'predicted_failure_probability',
            'num_gt',
            'num_detections',
            'tp',
            'fp',
            'fn',
            'mean_detection_confidence',
            'global_ood_score',
            'chip_ood_mean',
            'combined_ood_z',
        ]
    ].head(20)
)

In [ ]:
def show_image_class(row, *, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    image_id = row['image_id']
    class_id = int(row['class_id'])
    img = Image.open(eval_paths[image_id]).convert('RGB')
    ax.imshow(img)
    ax.axis('off')
    title = f"{row['dataset_name']} | {row['class_name']} | p(fail)={row.get('predicted_failure_probability', np.nan):.2f}"
    ax.set_title(title, fontsize=9)

    gt_rows = tables.ground_truth_enriched[
        (tables.ground_truth_enriched['image_id'] == image_id)
        & (tables.ground_truth_enriched['class_id'] == class_id)
    ]
    det_rows = det_enriched[
        (det_enriched['image_id'] == image_id)
        & (det_enriched['class_id'] == class_id)
    ]
    for _, g in gt_rows.iterrows():
        x1, y1, x2, y2 = g['bbox']
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', linewidth=2))
    for _, d in det_rows.iterrows():
        x1, y1, x2, y2 = d['bbox']
        color = 'cyan' if d['match_status'] == 'TP' else 'red'
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=2, linestyle='--'))
    return ax


gallery = risk[risk['dataset_name'].eq(SHIFTED_DATASET_NAME)].head(6)
if len(gallery) < 6:
    gallery = risk.head(6)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), gallery.reset_index(drop=True).iterrows()):
    show_image_class(row, ax=ax)
plt.tight_layout()
plt.show()

print('GT boxes are lime. TP detections are cyan dashed. FP detections are red dashed.')

## Limitations

- This result is specific to this detector, BDD100K reference data, and the clear-daytime vs shifted validation partition.
- The shifted BDD100K validation slice is useful but does not represent all real deployment shifts.
- The failure target is strict.
- Chip OOD depends on detector-predicted boxes, so fully missed objects may not have chip features.
- AUROC measures ranking, not calibration.

In this BDD100K validation-slice demo, OOD scores are not a replacement for detector confidence. They provide an additional risk signal for triaging likely failures from this detector under this specific driving-scene partition.